In [5]:
import re
import json
import base64
import requests

from datetime import datetime, timezone
from typing import List
from bs4 import BeautifulSoup, Tag
from dataclasses import dataclass

In [6]:
current_year = datetime.now(timezone.utc).year

headers = {
    "User-Agent": "Mozilla/5.0 (X11; Linux x86_64; rv:148.0) Gecko/20100101 Firefox/148.0"
}

### Formula 1 Scrapper

In [7]:
@dataclass
class SubEvent:
    day: str
    month: str
    title: str
    start_time: str
    end_time: str | None = None

    def __init__(self):
        return

In [8]:
@dataclass
class GroupEvent:
    link: str
    location: str
    title: str
    page: BeautifulSoup
    subevents: List[SubEvent]

    def __init__(self):
        return

In [9]:
def decodeBase64(data: str) -> str:
    return base64.b64decode(data).decode()

In [10]:
def mapToMonth(month_string: str) -> str:
    months = {
        "Jan": "01",
        "Feb": "02",
        "Mar": "03",
        "Apr": "04",
        "May": "05",
        "Jun": "06",
        "Jul": "07",
        "Aug": "08",
        "Sep": "09",
        "Oct": "10",
        "Nov": "11",
        "Dec": "12",
    }

    return months[month_string]

In [11]:
host = "www.formula1.com"

In [ ]:
try:
    url = f"https://{host}/en/racing/{current_year}"
    response = requests.get(url=url, headers=headers)
    response.raise_for_status()
    page = BeautifulSoup(response.text)
except requests.exceptions.RequestException as e:
    print(f"Error fetching URL: {e}")

In [13]:
events: List[GroupEvent] = []

In [14]:
calendar_container = page.find(
    "div",
    attrs={"data-f1rd-a7s-context": "eyJsb2NhdGlvbkluUGFnZSI6ImNhbGVuZGFyIn0="},
)

if calendar_container:
    grid_div = calendar_container.find("div", class_="grid")

    if grid_div:
        for link_tag in grid_div.children:
            if isinstance(link_tag, Tag):
                data_attr = link_tag.get("data-f1rd-a7s-context")

                if data_attr is None:
                    continue

                data = json.loads(decodeBase64("".join(data_attr)))

                link = link_tag.get("href")

                event = GroupEvent()

                event.link = "".join(link) if link else ""
                event.location = data.get("trackCountry")
                event.title = (
                    data.get("raceName")
                    .replace("FORMULA 1", "")
                    .replace("GRAND PRIX", "GP")
                    .replace("GRAN PREMIO", "GP")
                    .replace("GRANDE PRÊMIO", "GP")
                    .replace("2026", "")
                    .strip()
                )

                events.append(event)

In [ ]:
for event in events:
    try:
        url = f"https://{host}{event.link}"
        print(url)
        response = requests.get(url=url, headers=headers)
        response.raise_for_status()

        event.page = BeautifulSoup(response.text)
    except requests.exceptions.RequestException as e:
        print(f"Error fetching URL: {e}")

In [16]:
for event in events:
    event.subevents = []
    contents = event.page.find("div", class_="f1rd-page")

    if not contents:
        continue

    schedule_container = contents.find(attrs={"data-f1rd-a7s-context": True})

    if not schedule_container:
        continue

    data_attr = schedule_container.get("data-f1rd-a7s-context")
    if data_attr is None:
        continue

    data = json.loads(decodeBase64("".join(data_attr)))

    if not data or data.get("locationInPage") != "schedule":
        print("Not in 'schedule tag', skipping...")
        continue

    schedule_items = schedule_container.select("li[data-f1rd-a7s-context]")

    for item in schedule_items:
        direct_spans = item.find_all("span", recursive=False)

        if len(direct_spans) < 3:
            continue

        subevent = SubEvent()

        date_span = direct_spans[0]
        date_values = date_span.find_all("span", recursive=False)
        if len(date_values) >= 2:
            subevent.day = date_values[0].get_text(strip=True)
            subevent.month = mapToMonth(date_values[1].get_text(strip=True))

        info_span = direct_spans[2]
        info_children = info_span.find_all("span", recursive=False)
        if info_children:
            subevent.title = info_children[0].get_text(strip=True)

        time_tags = []

        if len(info_children) > 1:
            time_container = info_children[1]
            time_tags = time_container.find_all("time")

        if not time_tags and len(direct_spans) >= 4:
            time_span = direct_spans[3]
            time_tags = time_span.find_all("time")

        if not time_tags:
            time_tags = item.find_all("time")

        if time_tags:
            subevent.start_time = time_tags[0].get_text(strip=True)
            if len(time_tags) > 1:
                subevent.end_time = time_tags[1].get_text(strip=True)

        event.subevents.append(subevent)

In [ ]:
ical_headers = [
    "BEGIN:VCALENDAR",
    "PRODID:-//MotorSports Calendar//EN",
    "VERSION:2.0",
    "CALSCALE:GREGORIAN",
    "METHOD:PUBLISH"
]

In [28]:
event_entries: List[str] = []

### Calendar File Creation

In [19]:
from uuid import uuid4

In [29]:
for event in events:
    for subevent in event.subevents:
        year = datetime.now(timezone.utc).year
        month = subevent.month
        day = subevent.day

        start_time = subevent.start_time.split(":")
        start_hour = start_time[0]
        start_min = start_time[1]

        start_date_time = f"{year}{month}{day}T{start_hour}{start_min}00Z"
        end_date_time = None

        if subevent.end_time:
            end_time = subevent.end_time.split(":")
            end_hour = end_time[0]
            end_min = end_time[1]

            end_date_time = f"{year}{month}{day}T{end_hour}{end_min}00Z"

        timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

        event_entries.append("BEGIN:VEVENT")
        event_entries.append(f"SUMMARY:{subevent.title}")
        event_entries.append(f"CATEGORIES:{event.title}")
        event_entries.append(f"UID:{uuid4()}@motorsportscalendar")
        event_entries.append(f"DTSTAMP:{timestamp}")
        event_entries.append(f"DTSTART:{start_date_time}")
        if end_date_time:
            event_entries.append(f"DTEND:{end_date_time}")
        else:
            event_entries.append(f"DURATION:PT1H")
        event_entries.append("END:VEVENT")

In [30]:
ical_lines = [*ical_headers, *event_entries, "END:VCALENDAR"]

In [ ]:
with open("f1-cal.ics", "w", encoding="utf-8") as f:
    for line in ical_lines:
        f.write(line + "\r\n")